# rl_adv

> Phases 12-13, Track B: training returns and advantages of the RL loop (pure numpy, no GPU libraries): the slot's return minus the training-only penalties (invalid replies; with Phase 13's `wrongPenalty`, wrong answers), one advantage per board and slot against its group (Phase 12), or one per text turn from the reward-to-go against the group at the same slot and turn (Phase 13), and the training samples of a step.

In [ ]:
#| default_exp rl_adv

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import numpy as np


def wrongTimes(board, i):
    "Times of agent i's wrong answers, from the board's submit log (time, agent, subtask, candidates, result)."
    return [s[0] for s in board.submits if s[1] == i and s[4] == 'wrong']


def trainingReturns(ro, penalty, wrongPenalty=0.0):
    """Rollout returns minus `penalty` per invalid reply of the slot and `wrongPenalty` per wrong answer of the slot (from
    the board's submit log; training only: the reported welfare and the turn logs are unchanged)."""
    R = np.array(ro.returns, float)
    for b, lg in enumerate(ro.logs):
        for i, recs in enumerate(lg):
            R[b, i] -= penalty * sum(not x['valid'] for x in recs)
            if wrongPenalty:
                R[b, i] -= wrongPenalty * len(wrongTimes(ro.boards[b], i))
    return R


def groupAdvantages(returns):
    "returns (seeds x group x N) -> per slot, (R - group mean) / (group std + 1e-6); 0 where the group's returns are equal."
    R = np.asarray(returns, float)
    mu, sd = R.mean(axis=1, keepdims=True), R.std(axis=1, keepdims=True)
    return np.where(sd > 1e-9, (R - mu) / (sd + 1e-6), 0.0)


def turnRewards(ro, penalty, wrongPenalty=0.0):
    """Per board and slot, the list of the slot's per-turn training rewards: the reward delivered at the turn (turn log),
    minus `penalty` for an invalid reply and `wrongPenalty` for a wrong answer at that turn (matched by its time)."""
    out = []
    for b, lg in enumerate(ro.logs):
        out.append([])
        for i, recs in enumerate(lg):
            wrong = set(wrongTimes(ro.boards[b], i)) if wrongPenalty else set()
            out[b].append([float(x['reward']) - penalty * (not x['valid']) - wrongPenalty * (float(x['t']) in wrong)
                           for x in recs])
    return out


def rewardsToGo(ro, penalty, wrongPenalty=0.0):
    """Per board, slot and text turn, G_t: the slot's per-turn rewards from turn t on plus the credit still pending for it
    at the end (as the rollout's returns), so G at the first turn is the slot's training return."""
    return [[list(np.cumsum(r[::-1])[::-1] + float(ro.boards[b].pending[i])) if r else [] for i, r in enumerate(rb)]
            for b, rb in enumerate(turnRewards(ro, penalty, wrongPenalty))]


def turnAdvantages(ro, penalty, wrongPenalty=0.0):
    """Per board, slot and text turn: (G_t - group mean) / (group std + 1e-6), the group being the boards of the same
    seed (the G replays of one board) at the same slot and turn index; 0 where the group's values are equal."""
    G = rewardsToGo(ro, penalty, wrongPenalty)
    A = [[[0.0] * len(g) for g in Gb] for Gb in G]
    groups = {}
    for b, s in enumerate(ro.seeds):
        groups.setdefault(s, []).append(b)
    for bs in groups.values():
        for i in range(len(G[bs[0]])):
            for n in range(max(len(G[b][i]) for b in bs)):
                have = [b for b in bs if n < len(G[b][i])]
                v = np.array([G[b][i][n] for b in have])
                if v.std() > 1e-9:
                    for b, x in zip(have, v):
                        A[b][i][n] = float((x - v.mean()) / (v.std() + 1e-6))
    return A


def stepSamples(ro, adv, fraction=1.0, rng=None):
    """One sample per text turn with a nonzero advantage (slot, prompt, reply text, cut or not, advantage); `adv` per board
    and slot (one value for all of the slot's turns, Phase 12) or per board, slot and turn (a list per board and slot);
    with fraction < 1, a random share of them (an unbiased, cheaper estimate of the same gradient)."""
    out = []
    for b in range(len(ro.boards)):
        for i, (recs, ps) in enumerate(zip(ro.logs[b], ro.prompts[b])):
            a = adv[b][i]
            for n, (x, p) in enumerate(zip(recs, ps)):
                v = float(a[n] if np.ndim(a) else a)
                if v != 0:
                    out.append(dict(slot=i, system=p[0], blocks=p[1], reply=x['response'], eos=x['stop'] != 'max_tokens', adv=v))
    if fraction >= 1 or not out:
        return out
    keep = sorted((rng or np.random.default_rng(0)).choice(len(out), max(1, round(fraction * len(out))), replace=False))
    return [out[k] for k in keep]

## Tests

In [ ]:
import copy
from types import SimpleNamespace
from explRL.clue_policy import claudePolicy
from explRL.clue_roll import rollout
from explRL.llm_client import LLMClient
from explRL.llm_fake import FakeClient, fakeReply
from explRL.llm_runs import LLMConfig

# AC-5 (Phase 12): group advantages on hand-made returns (seeds x group x N); equal returns give 0
R = np.array([[[1.0, 0.0], [3.0, 0.0]], [[2.0, 5.0], [2.0, 1.0]]])
A = groupAdvantages(R)
assert np.allclose(A[0, :, 0], [-1, 1], atol=1e-5) and np.all(A[0, :, 1] == 0) and np.all(A[1, :, 0] == 0)
assert np.allclose(A[1, :, 1], [1, -1], atol=1e-5)
# the format penalty lowers training returns only, per invalid reply of the slot
ro = SimpleNamespace(returns=np.array([[1.0, 2.0]]), logs=[[[dict(valid=True), dict(valid=False)], [dict(valid=False)] * 3]])
assert np.allclose(trainingReturns(ro, 0.1), [[0.9, 1.7]]) and np.allclose(ro.returns, [[1.0, 2.0]])

# hand-made boards (AC-4): turn records (time, delivered reward, valid), the board's submit log and the credit pending at the end
rec = lambda t, r, valid=True: dict(t=t, reward=r, valid=valid)
bd = lambda pending, submits=(): SimpleNamespace(pending=np.array(pending, float), submits=list(submits))
def ro2(logs, boards, seeds):
    ret = [[sum(x['reward'] for x in recs) + b.pending[i] for i, recs in enumerate(lg)] for lg, b in zip(logs, boards)]
    return SimpleNamespace(logs=logs, boards=boards, seeds=seeds, returns=np.array(ret))
# wrong answers cost wrongPenalty in training returns only (from the submit log: slot 0 twice, slot 1 once; 'late' is free)
subs = [(0.0, 0, 3, 2, 'wrong'), (2.0, 0, 3, 1, 'wrong'), (1.0, 1, 4, None, 'wrong'), (3.0, 1, 4, 1, 'late')]
r = ro2([[[rec(0.0, -0.5), rec(2.0, -0.5, False)], [rec(1.0, -0.5), rec(3.0, 0.0)]]], [bd([0, 0], subs)], [7])
logs0 = copy.deepcopy(r.logs)
assert np.allclose(trainingReturns(r, 0.1, 2.5), [[-1 - 0.1 - 5, -0.5 - 2.5]]) and np.allclose(trainingReturns(r, 0.1), [[-1.1, -0.5]])
assert np.array_equal(trainingReturns(r, 0.1, 0.0), trainingReturns(r, 0.1)) and r.logs == logs0      # turn logs untouched
# per-turn rewards: the wrong answer at its own turn (matched by time), the format penalty at the invalid turn
assert np.allclose(turnRewards(r, 0.1, 2.5)[0][0], [-3.0, -3.1]) and np.allclose(turnRewards(r, 0.1, 2.5)[0][1], [-3.0, 0.0])
assert turnRewards(r, 0.1)[0][1] == [-0.5, 0.0]
# equal groups -> 0 at every turn
same = ro2([[[rec(0, 1.0), rec(1, 0.0)]]] * 3, [bd([0.5])] * 3, [5, 5, 5])
assert turnAdvantages(same, 0.1, 2.5) == [[[0.0, 0.0]]] * 3
# a late credit counts toward the sharing turn's reward-to-go: board 0's slot 0 shares at turn 0 (reward 0) and is credited
# at turn 2 (delivered) or only at the end (pending); the replays where nothing is credited get the opposite advantage
for logs, pend in (([rec(0, 0.0), rec(1, 0.0), rec(2, 1.5)], 0.0), ([rec(0, 0.0), rec(1, 0.0), rec(2, 0.0)], 1.5)):
    t = ro2([[logs], [[rec(0, 0.0), rec(1, 0.0), rec(2, 0.0)]]], [bd([pend]), bd([0.0])], [9, 9])
    G, a = rewardsToGo(t, 0.1), turnAdvantages(t, 0.1)
    assert np.allclose(G[0][0], [1.5, 1.5, 1.5]) and np.allclose(a[0][0], [1, 1, 1], atol=1e-5) and np.allclose(a[1][0], [-1, -1, -1], atol=1e-5)
# per turn, not per board: the same total, earned at different turns, differs only at the turn in between
t = ro2([[[rec(0, 0.0), rec(1, 2.0), rec(2, 0.0)]], [[rec(0, 0.0), rec(1, 0.0), rec(2, 2.0)]]], [bd([0.0])] * 2, [3, 3])
a = turnAdvantages(t, 0.1)
assert a[0][0][:2] == [0.0, 0.0] and np.allclose([a[0][0][2], a[1][0][2]], [-1, 1], atol=1e-5)
assert np.all(groupAdvantages(trainingReturns(t, 0.1).reshape(1, 2, 1)) == 0)     # the board-level advantage sees no difference
# groups are the boards of one seed, whatever their order
t = ro2([[[rec(0, 1.0)]], [[rec(0, 0.0)]], [[rec(0, 3.0)]], [[rec(0, 2.0)]]], [bd([0.0])] * 4, [1, 2, 1, 2])
assert np.allclose([x[0][0] for x in turnAdvantages(t, 0.1)], [-1, -1, 1, 1], atol=1e-5)

# samples: one per text turn of slots with nonzero advantage, with its prompt; zero-advantage slots skipped
cfg = LLMConfig(task='clue', channel='blackboard', N=4, turns=3, world=dict(m=2, r=1, R=30))
ro = rollout(cfg, [900, 900], claudePolicy(LLMClient('claude-haiku-5-5', client=FakeClient(fakeReply))), keepPrompts=True)
adv = np.zeros((2, 4)); adv[0, 1], adv[1, 3] = 0.5, -1.0
smp = stepSamples(ro, adv)
assert len(smp) == 6 and {x['slot'] for x in smp} == {1, 3} and all(x['blocks'][-1]['text'].startswith('Now:\n') for x in smp)
assert [x['adv'] for x in smp] == [0.5] * 3 + [-1.0] * 3 and all(x['eos'] for x in smp)
# a share of the turns: a seeded, sorted subset of the full set, of the requested size
full = stepSamples(ro, adv)
part = stepSamples(ro, adv, 0.5, np.random.default_rng([0, 7]))
assert len(part) == 3 and all(x in full for x in part) and part == stepSamples(ro, adv, 0.5, np.random.default_rng([0, 7]))
assert stepSamples(ro, adv * 0, 0.25) == [] and len(stepSamples(ro, adv, 0.01)) == 1
# per-turn advantages: a constant list per slot gives the per-slot samples; zero turns are skipped
perTurn = [[[float(adv[b, i])] * len(ro.logs[b][i]) for i in range(4)] for b in range(2)]
assert stepSamples(ro, perTurn) == full and stepSamples(ro, perTurn, 0.5, np.random.default_rng([0, 7])) == part
perTurn[0][1][1] = 0.0
assert [x['adv'] for x in stepSamples(ro, perTurn)] == [0.5, 0.5] + [-1.0] * 3
# on a real rollout: G at the first turn is the training return; the board's wrong answers are the turn logs' wrong submits
rt = trainingReturns(ro, 0.1, 2.5)
assert all(np.isclose(rewardsToGo(ro, 0.1, 2.5)[b][i][0], rt[b, i]) for b in range(2) for i in range(4))
assert sum(len(wrongTimes(b, i)) for b in ro.boards for i in range(4)) == sum(b.counts['wrong'] for b in ro.boards) > 0
ta = turnAdvantages(ro, 0.1, 2.5)
assert [len(x) for x in ta[0]] == [len(x) for x in ro.logs[0]] and all(v == 0 for b in ta for x in b for v in x)  # replays of one board, same replies

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()